In [158]:
!pip install gensim


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [159]:
!pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [160]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
import re

import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, HTML
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

In [161]:
ev2 = pd.read_csv("../../Datasets/evaluacion2.csv")
ev2.head(3)

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,5.484755,0.000000,5.883322
1,OWC 250GB Aura Pro 6G Flash SSD Upgrade for 20...,4.6,No Badge,Sponsored,1,0,0,0.0,Storage & Memory Cards,Media,3.783962,0.000000,5.624018
2,HP 67XL Black High-yield Ink Cartridge | Works...,4.6,Best Seller,Organic,0,1,0,0.0,"Office Supplies, Ink & Toner",Media,3.607941,10.819798,11.523598


In [162]:
ev2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   product_title             7717 non-null   object 
 1   product_rating            7717 non-null   float64
 2   is_best_seller            7717 non-null   object 
 3   is_sponsored              7717 non-null   object 
 4   buy_box_availability      7717 non-null   int64  
 5   sustainability_tags       7717 non-null   int64  
 6   has_coupon                7717 non-null   int64  
 7   discount_percentage       7717 non-null   float64
 8   product_category          7717 non-null   object 
 9   product_segment           7717 non-null   object 
 10  log_original_price        7717 non-null   float64
 11  log_purchased_last_month  7717 non-null   float64
 12  log_total_reviews         7717 non-null   float64
dtypes: float64(5), int64(3), object(5)
memory usage: 783.9+ KB


In [163]:
# Preparar títulos para Word2Vec
def preprocess_titles(titles):
    processed = []
    for title in titles:
        # Limpiar texto
        text = str(title).lower()
        # Remover caracteres especiales pero mantener palabras
        text = re.sub(r'[^\w\s]', ' ', text)
        # Tokenizar
        tokens = simple_preprocess(text)
        processed.append(tokens)
    return processed

# Procesar los títulos
titles = ev2['product_title'].tolist()
processed_titles = preprocess_titles(titles)

In [164]:
# Entrenar modelo Word2Vec
model = Word2Vec(
    sentences=processed_titles,
    vector_size=100,  # dimensión de los vectores
    window=5,  # contexto
    min_count=2,  # palabras que aparecen al menos 2 veces
    workers=4,
    epochs=30
)

# Guardar el modelo
# model.save("product_titles_word2vec.model")

In [165]:
# known_brands = [
#     # Principales fabricantes de PCs/Laptops
#     'hp', 'hewlett packard', 'dell', 'lenovo', 'apple', 'asus', 'acer',
#     'msi', 'toshiba', 'samsung', 'lg', 'microsoft', 'google', 'huawei',
#     'razer', 'alienware', 'framework', 'system76', 'chuwi', 'vaio',
    
#     # Procesadores y chips
#     'intel', 'amd', 'qualcomm', 'nvidia', 'arm', 'mediatek', 'apple silicon',
#     'apple m1', 'apple m2', 'apple m3', 'ryzen', 'core i3', 'core i5',
#     'core i7', 'core i9', 'xeon', 'athlon', 'celeron', 'pentium',
#     'snapdragon', 'exynos', 'tegra',
    
#     # Tarjetas gráficas
#     'nvidia', 'geforce', 'rtx', 'gtx', 'quadro', 'tesla', 'amd', 'radeon',
#     'intel arc', 'asus rog', 'msi gaming', 'gigabyte', 'evga', 'zotac',
#     'sapphire', 'xfx', 'powercolor',
    
#     # Almacenamiento
#     'samsung', 'wd', 'western digital', 'seagate', 'toshiba', 'crucial',
#     'kingston', 'sandisk', 'adata', 'team group', 'silicon power',
#     'transcend', 'pny', 'intel ssd', 'sabrent', 'corsair', 'gigabyte',
#     'patriot', 'ocz', 'plextor', 'micron', 'sk hynix', 'lexar',
    
#     # Memoria RAM
#     'corsair', 'gskill', 'kingston', 'crucial', 'team group', 'adata',
#     'samsung', 'micron', 'patriot', 'oloy', 'silicon power', 'geil',
#     'transcend', 'hp memory', 'dell memory',
    
#     # Motherboards
#     'asus', 'gigabyte', 'msi', 'asus rog', 'asus tuf', 'asus prime',
#     'gigabyte aorus', 'msi mpg', 'msi mag', 'asusrock', 'biostar',
#     'evga', 'intel', 'super micro', 'tyan',
    
#     # Fuentes de alimentación
#     'corsair', 'evga', 'seasonic', 'cooler master', 'be quiet', 'thermaltake',
#     'asus rog', 'gigabyte', 'msi', 'nzxt', 'fsp', 'super flower', 'antec',
#     'deepcool', 'silverstone', 'ocz', 'xfx',
    
#     # Refrigeración
#     'corsair', 'nzxt', 'cooler master', 'noctua', 'be quiet', 'deepcool',
#     'thermaltake', 'arctic', 'cryorig', 'fractal design', 'phanteks',
#     'ekwb', 'lian li', 'id cooling', 'scythe', 'thermalright',
    
#     # Gabinetes/Cases
#     'nzxt', 'corsair', 'fractal design', 'lian li', 'cooler master',
#     'phanteks', 'thermaltake', 'be quiet', 'deepcool', 'in win',
#     'silverstone', 'antec', 'asus rog', 'msi', 'gigabyte', 'aerocool',
    
#     # Monitores
#     'asus', 'dell', 'samsung', 'lg', 'acer', 'benq', 'msi', 'viewsonic',
#     'aoc', 'hp', 'lenovo', 'gigabyte', 'philips', 'acer nitro',
#     'asus rog', 'asus tuf', 'alienware', 'apple studio display',
#     'apple pro display xdr', 'huawei', 'xiaomi',
    
#     # Periféricos (Teclados, Ratones, etc.)
#     'logitech', 'razer', 'corsair', 'steelseries', 'hyperx', 'asus rog',
#     'msi', 'redragon', 'cooler master', 'gskill', 'finalmouse',
#     'zowie', 'glorious', 'ducky', 'keychron', 'anne pro', 'leopold',
#     'filco', 'varmilo', 'iqunix',
    
#     # Audio
#     'logitech', 'razer', 'corsair', 'steelseries', 'hyperx', 'asus rog',
#     'jbl', 'bose', 'sony', 'sennheiser', 'audio technica', 'beyerdynamic',
#     'akg', 'shure', 'hyperx cloud', 'steel series arctis', 'jabra',
#     'plantronics', 'poly', 'creative', 'edifier', 'marshall',
    
#     # Impresoras y Escáneres
#     'hp', 'canon', 'epson', 'brother', 'lexmark', 'xerox', 'ricoh',
#     'kyocera', 'samsung', 'dell', 'okidata', 'sharp', 'panasonic',
#     'fujitsu', 'kodak', 'hp laserjet', 'hp officejet', 'hp deskjet',
    
#     # Redes y WiFi
#     'tp link', 'netgear', 'asus', 'linksys', 'd link', 'ubiquiti',
#     'mikrotik', 'cisco', 'aruba', 'tplink', 'meraki', 'ruckus',
#     'zyxel', 'trendnet', 'synology', 'qnap',
    
#     # NAS y Storage empresarial
#     'synology', 'qnap', 'western digital', 'seagate', 'netgear',
#     'asus', 'terra master', 'buffalo', 'd link', 'lenovo emc',
#     'dell emc', 'hp storage', 'ibm storage',
    
#     # Componentes Apple/Mac
#     'owc', 'other world computing', 'macsales', 'belkin', 'anker',
#     'satechi', 'hyper', 'caldigit', 'owc thunderbay', 'sonnet',
    
#     # Accesorios móviles
#     'anker', 'belkin', 'spigen', 'otterbox', 'mophie', 'ugreen',
#     'aukey', 'ravpower', 'xiaomi', 'samsung accessories',
#     'apple accessories', 'google accessories',
    
#     # Gaming
#     'nintendo', 'playstation', 'xbox', 'steam deck', 'valve',
#     'oculus', 'meta quest', 'htc vive', 'valve index',
#     'pimax', 'hp reverb', 'playstation vr',
    
#     # Smartphones
#     'apple iphone', 'samsung galaxy', 'google pixel', 'oneplus',
#     'xiaomi', 'huawei', 'oppo', 'vivo', 'motorola', 'nokia',
#     'sony xperia', 'asus zenfone', 'lg g series', 'lg v series',
    
#     # Tablets
#     'apple ipad', 'samsung galaxy tab', 'microsoft surface',
#     'lenovo tab', 'huawei matepad', 'amazon fire', 'google pixel tablet',
#     'xiaomi pad', 'asus zenpad',
    
#     # Smartwatches y Wearables
#     'apple watch', 'samsung galaxy watch', 'fitbit', 'garmin',
#     'huawei watch', 'xiaomi mi band', 'amazfit', 'fossil',
#     'misfit', 'withings', 'suunto',
    
#     # Cámaras y Fotografía
#     'canon', 'nikon', 'sony', 'fujifilm', 'panasonic', 'olympus',
#     'gopro', 'dji', 'insta360', 'ricoh', 'pentax', 'leica',
#     'sigma', 'tamron', 'tokina', 'zeiss', 'sam yang',
    
#     # Proyectores
#     'epson', 'benq', 'optoma', 'viewsonic', 'lg', 'samsung',
#     'sony', 'acer', 'xiaomi', 'vankyo', 'apeman',
    
#     # Software y SaaS
#     'microsoft', 'adobe', 'autodesk', 'vmware', 'citrix',
#     'oracle', 'ibm', 'salesforce', 'sap', 'servicenow',
#     'workday', 'atlassian', 'slack', 'zoom', 'teams',
    
#     # Energía y UPS
#     'apc', 'cyberpower', 'tripp lite', 'eaton', 'vertiv',
#     'schneider electric', 'belkin', 'apc by schneider',
    
#     # Marcas de retail/tiendas
#     'best buy', 'geek squad', 'micro center', 'newegg', 'amazon basics',
#     'walmart onn', 'costco kirkland', 'monoprice', 'insignia',
#     'dynex', 'on', 'onn',
    
#     # Marcas misceláneas
#     '3m', 'velcro', 'command', 'scotch', 'kensington', 'targus',
#     'uni', 'iflash', 'ifixit', 'tecknet', 'j5create',
    
#     # Marcas por tipo de producto específico
#     # Ink/Toner
#     'hp ink', 'hp toner', 'brother ink', 'brother toner',
#     'canon ink', 'canon toner', 'epson ink', 'epson toner',
#     'lexmark toner', 'generic ink', 'compatible ink',
    
#     # Cables y adaptadores
#     'anker', 'ugreen', 'cable matters', 'uni', 'j5create',
#     'plugable', 'startech', 'cablecreation', 'ivanky',
    
#     # Sillas gaming/oficina
#     'secretlab', 'dxracer', 'akracing', 'noblechairs',
#     'autonomous', 'herman miller', 'steelcase', 'humanscale',
    
#     # Escritorios
#     'uplift desk', 'fully', 'autonomous', 'ikea', 'varidesk',
    
#     # Abreviaciones comunes
#     'wd', 'nv', 'amd', 'intel', 'nvidia', 'msi', 'asus',
#     'gigabyte', 'corsair', 'kingston', 'crucial', 'adata',
#     'teamgroup', 'gskill', 'noctua', 'bequiet', 'arctic',
#     'deepcool', 'phanteks', 'lianli', 'fractal',
# ]

In [ ]:
# Lista de marcas conocidas en tu categoría 
known_brands = ['hp', 'dell', 'lenovo', 'samsung', 'apple', 'lg', 'sony', 
                'microsoft', 'logitech', 'canon', 'nvidia', 'intel', 'amd', 'texas',
                'owc', 'vivo', 'seagate', 'western digital', 'kingston', 'crucial', 'lorex',
                  'monoprice', 'amazon', 'duracell', 'roku', 'scotch', 'sharpie', 'tp-link', 'jbl']

# Función para encontrar marca en un título
def extract_brand_from_title(title, brand_list=known_brands, threshold=0.5):
    title_tokens = simple_preprocess(str(title).lower())
    
    best_match = None
    best_score = 0
    
    for token in title_tokens:
        if len(token) < 3:  # Ignorar tokens muy cortos
            continue
            
        for brand in brand_list:
            brand_words = brand.lower().split()
            
            # Verificación exacta
            if any(brand_word in token or token in brand_word for brand_word in brand_words):
                return brand.title()
            
            # Verificación usando Word2Vec 
            try:
                if token in model.wv:
                    similarity = max([model.wv.similarity(token, bw) for bw in brand_words if bw in model.wv])
                    if similarity > threshold and similarity > best_score:
                        best_score = similarity
                        best_match = brand
                        print(best_score)
            except:
                continue
    
    return best_match.title() if best_match else "Unknown"

# Aplicar a todo el dataset
ev2['extracted_brand'] = ev2['product_title'].apply(extract_brand_from_title)

0.5505338
0.5728151
0.5204962
0.5728151
0.675314
0.51215255
0.5728151
0.56622773
0.5934256
0.53746486
0.6332739
0.51435596
0.5500636
0.5505338
0.87757295
0.5204962
0.6978611
0.5505338
0.58137435
0.56622773
0.5107379
0.56905234
0.60442847
0.6252877
0.6488592
0.55434686
0.56905234
0.8095636
0.56471705
0.6121169
0.63143104
0.5891307
0.54121304
0.5500636
0.6065185
0.66630757
0.5204962
0.6143592
0.50014436
0.5005982
0.50503874
0.572073
0.57438374
0.5654528
0.56046784
0.5389488
0.5500636
0.50494486
0.5226552
0.545114
0.8090871
0.5366956
0.5618825
0.5980516
0.5505338
0.5204962
0.5934256
0.51215255
0.5618825
0.5591778
0.63572973
0.51215255
0.6446001
0.5336575
0.54811305
0.6505026
0.56622773
0.56622773
0.513111
0.5891307
0.5311406
0.72712845
0.5505338
0.51215255
0.5482435
0.72416717
0.8090871
0.81345236
0.8206169
0.50494486
0.5226552
0.6277976
0.5540847
0.56622773
0.5944844
0.6268544
0.6394574
0.5505338
0.5311406
0.72712845
0.5719619
0.5728151
0.5505338
0.5482435
0.72416717
0.8090871
0.8206169


In [ ]:
ev2.head()

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews,extracted_brand
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,5.484755,0.000000,5.883322,Owc
1,OWC 250GB Aura Pro 6G Flash SSD Upgrade for 20...,4.6,No Badge,Sponsored,1,0,0,0.0,Storage & Memory Cards,Media,3.783962,0.000000,5.624018,Owc
2,HP 67XL Black High-yield Ink Cartridge | Works...,4.6,Best Seller,Organic,0,1,0,0.0,"Office Supplies, Ink & Toner",Media,3.607941,10.819798,11.523598,Tp-Link
3,HP 67 Black/Tri-color Ink Cartridges for HP Pr...,4.6,No Badge,Organic,0,0,0,0.0,"Office Supplies, Ink & Toner",Media,3.804215,10.819798,10.986868,Tp-Link
4,"Sony ZX Series Wired On-Ear Headphones, Black ...",4.5,Best Seller,Organic,0,0,0,0.0,"Audio, Sound & Recording Gear",Baja,2.355178,9.210440,11.598433,Sony


In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Preparar datos
# Si quieres usar One-Hot Encoding, hazlo para todas las categóricas
categorical_cols = ['is_best_seller', 'is_sponsored', 'product_category', 
                    'product_segment']

# Aplicar One-Hot Encoding a todas las variables categóricas
ev2_encoded = pd.get_dummies(ev2, columns=categorical_cols + ['extracted_brand'], 
                            drop_first=True)

# Preparar X e y
X = ev2_encoded.drop(columns=['product_title', 'log_original_price'])
y = ev2_encoded['log_original_price']

# Verificar tipos de datos
print("Tipos de datos en X:")
print(X.dtypes.value_counts())

# Split de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modelo XGBoost - SIN enable_categorical porque ya usamos One-Hot
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=20,
    eval_metric='rmse'
)

# Entrenar con validación temprana
print("Entrenando modelo...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100  # Para ver progreso
)

# Predicciones
y_pred = model.predict(X_test)

# Convertir de log a escala original
y_test_eur = np.expm1(y_test)
y_pred_eur = np.expm1(y_pred)

# Métricas
r2 = r2_score(y_test_eur, y_pred_eur)
rmse = np.sqrt(mean_squared_error(y_test_eur, y_pred_eur))
mae = np.mean(np.abs(y_test_eur - y_pred_eur))

print(f'\n=== RESULTADOS ===')
print(f'R² Score: {r2:.4f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAE: {mae:.2f}')
print(f'Rango de precios real: €{y_test_eur.min():.2f} - €{y_test_eur.max():.2f}')

# Importancia de features
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nTop 10 features más importantes:')
print(feature_importance.head(10))

# Mostrar primeras filas
print('\nPrimeras filas de X:')
print(X.head())

Tipos de datos en X:
bool       60
float64     4
int64       3
Name: count, dtype: int64
Entrenando modelo...
[0]	validation_0-rmse:1.25664
[100]	validation_0-rmse:0.50325
[200]	validation_0-rmse:0.49352
[270]	validation_0-rmse:0.49323

=== RESULTADOS ===
R² Score: 0.7739
RMSE: 205.08
MAE: 86.69
Rango de precios real: €3.98 - €4995.95

Top 10 features más importantes:
                                              feature  importance
36                               product_segment_Baja    0.399932
37                              product_segment_Media    0.107837
27                 product_category_Power & Batteries    0.059667
19       product_category_Chargers, Adapters & Cables    0.050283
21                           product_category_Laptops    0.045819
29  product_category_Small Gadget Accessories (Cas...    0.043788
18             product_category_Cameras & Photography    0.025784
28    product_category_Printers & Scanners (Hardware)    0.020418
5                            log_pu

In [ ]:
ev2.head(1)

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,...,brand_Samsung,brand_Scotch,brand_Seagate,brand_Sharpie,brand_Sony,brand_Texas,brand_Tp-Link,brand_Unknown,brand_Vivo,brand_Western Digital
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
ev2.to_csv('ev2_Word2vec.csv', index=False)